# Streamlit Application

This phase focuses on developing an interactive web application for the AI-Based Student Placement Prediction and Career Recommendation System.

The application will allow students to enter their academic and skill details, generate placement predictions and career recommendations, and store the results in the MySQL database.

## Streamlit Setup

In this section, the Streamlit environment and required libraries are checked before developing the application.
The application will use the trained machine learning model, preprocessing components, career recommendation logic, and MySQL database.

In [17]:
import streamlit as st
import joblib
import pandas as pd

In [11]:
print("Streamlit version:", st.__version__)
print("Streamlit imported successfully!")

Streamlit version: 1.61.1
Streamlit imported successfully!


## Application Layout

The Streamlit application layout is designed to provide a simple and user-friendly interface for student placement prediction and career recommendation.

The application will contain sections for student information, academic and skill details, prediction results, career recommendations, and database storage.

## Student Input Form

The Streamlit application collects the student's personal, academic, and technical information required by the placement prediction model.

The form accepts the 16 original input features used during model development. Engineered features and prediction results are calculated automatically by the system.

In [12]:
st.title("🎓 AI-Based Student Placement Prediction")
st.subheader("Student Information")

student_name = st.text_input("Student Name")

branch = st.selectbox(
    "Branch",
    ["CSE", "IT", "ECE", "EEE", "Mechanical", "Civil", "Other"]
)

college_tier = st.selectbox(
    "College Tier",
    ["Tier-1", "Tier-2", "Tier-3"]
)

st.subheader("📚 Academic Details")

cgpa = st.number_input(
    "CGPA",
    min_value=0.0,
    max_value=10.0,
    value=7.0,
    step=0.1
)

backlogs = st.number_input(
    "Number of Backlogs",
    min_value=0,
    max_value=20,
    value=0,
    step=1
)

st.subheader("💻 Technical & Skill Details")

coding_skills = st.slider(
    "Coding Skills",
    min_value=0.0,
    max_value=10.0,
    value=5.0,
    step=0.5
)

dsa_score = st.slider(
    "DSA Score",
    min_value=0.0,
    max_value=10.0,
    value=5.0,
    step=0.5
)

aptitude_score = st.slider(
    "Aptitude Score",
    min_value=0.0,
    max_value=10.0,
    value=5.0,
    step=0.5
)

communication_skills = st.slider(
    "Communication Skills",
    min_value=0.0,
    max_value=10.0,
    value=5.0,
    step=0.5
)

ml_knowledge = st.slider(
    "Machine Learning Knowledge",
    min_value=0.0,
    max_value=10.0,
    value=5.0,
    step=0.5
)

system_design = st.slider(
    "System Design Knowledge",
    min_value=0.0,
    max_value=10.0,
    value=5.0,
    step=0.5
)

st.subheader("🏆 Experience & Activities")

internships = st.number_input(
    "Number of Internships",
    min_value=0,
    max_value=10,
    value=0,
    step=1
)

projects_count = st.number_input(
    "Number of Projects",
    min_value=0,
    max_value=20,
    value=0,
    step=1
)

certifications = st.number_input(
    "Number of Certifications",
    min_value=0,
    max_value=50,
    value=0,
    step=1
)

hackathons = st.number_input(
    "Number of Hackathons",
    min_value=0,
    max_value=50,
    value=0,
    step=1
)

open_source_contributions = st.number_input(
    "Open Source Contributions",
    min_value=0,
    max_value=100,
    value=0,
    step=1
)

extracurriculars = st.number_input(
    "Extracurricular Activities",
    min_value=0,
    max_value=50,
    value=0,
    step=1
)

2026-08-15 20:01:36.530 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-15 20:01:36.532 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-15 20:01:36.533 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-15 20:01:36.535 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-15 20:01:36.537 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-15 20:01:36.538 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-15 20:01:36.539 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-15 20:01:36.540 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

## Apply Feature Engineering

The Streamlit application collects the student's original features through the input form.

The four engineered features used during model training are not entered manually by the user. Instead, they are calculated automatically using the same feature-engineering logic used during model development.

The engineered features are:

- `technical_skill_score`
- `experience_score`
- `has_backlog`
- `technical_skill_gap`

This ensures that new student data has the same feature structure as the data used to train the machine learning model.

In [13]:
technical_skill_score = (
    coding_skills
    + dsa_score
    + ml_knowledge
    + system_design
) / 4

experience_score = (
    internships
    + projects_count
    + certifications
    + hackathons
    + open_source_contributions
    + extracurriculars
)

has_backlog = 1 if backlogs > 0 else 0

maximum_skill = max(
    coding_skills,
    dsa_score,
    ml_knowledge,
    system_design
)

minimum_skill = min(
    coding_skills,
    dsa_score,
    ml_knowledge,
    system_design
)

technical_skill_gap = maximum_skill - minimum_skill

In [14]:
{
    "technical_skill_score": technical_skill_score,
    "experience_score": experience_score,
    "has_backlog": has_backlog,
    "technical_skill_gap": technical_skill_gap
}

{'technical_skill_score': 5.0,
 'experience_score': 0,
 'has_backlog': 0,
 'technical_skill_gap': 0.0}

## Generate Placement Prediction

The engineered student features are passed through the fitted preprocessing pipeline and then provided to the trained Gradient Boosting classification model.

The model predicts the student's placement status:

- `1` → Placed
- `0` → Not Placed

The trained model and preprocessing pipeline developed during the machine learning phase are reused in the Streamlit application.

### Load Trained Model and Preprocessor

The saved preprocessing pipeline and trained Gradient Boosting model are loaded from the project's `models` directory.

These components are reused to generate predictions for new student profiles.

In [15]:
preprocessor = joblib.load("../models/preprocessor.pkl")
best_model = joblib.load("../models/best_model.pkl")

type(preprocessor), type(best_model)

(sklearn.compose._column_transformer.ColumnTransformer,
 sklearn.ensemble._gb.GradientBoostingClassifier)

### Prepare Student Features for Prediction

The student's manually entered values and automatically calculated engineered features are combined into a Pandas DataFrame.

The target variable `placement_status` is not included because it is the value that the machine learning model must predict.

In [18]:
student_data = {
    "branch": branch,
    "college_tier": college_tier,
    "cgpa": cgpa,
    "backlogs": backlogs,
    "coding_skills": coding_skills,
    "dsa_score": dsa_score,
    "aptitude_score": aptitude_score,
    "communication_skills": communication_skills,
    "ml_knowledge": ml_knowledge,
    "system_design": system_design,
    "internships": internships,
    "projects_count": projects_count,
    "certifications": certifications,
    "hackathons": hackathons,
    "open_source_contributions": open_source_contributions,
    "extracurriculars": extracurriculars,
    "technical_skill_score": technical_skill_score,
    "has_backlog": has_backlog,
    "experience_score": experience_score,
    "technical_skill_gap": technical_skill_gap
}

student_df = pd.DataFrame([student_data])

student_df

,branch,college_tier,cgpa,backlogs,coding_skills,dsa_score,aptitude_score,communication_skills,ml_knowledge,system_design,internships,projects_count,certifications,hackathons,open_source_contributions,extracurriculars,technical_skill_score,has_backlog,experience_score,technical_skill_gap
0,CSE,Tier-1,7.0,0,5.0,5.0,5.0,5.0,5.0,5.0,0,0,0,0,0,0,5.0,0,0,0.0


### Generate Placement Prediction

The prepared student data is transformed using the same fitted preprocessing pipeline used during model training.

The transformed data is then passed to the trained Gradient Boosting model.

The resulting prediction is converted into a human-readable placement status.

In [20]:
student_encoded = preprocessor.transform(student_df)
student_encoded.shape

(1, 28)

In [21]:
prediction = best_model.predict(student_encoded)
prediction

array([0])

In [22]:
placement_status = "Placed" if prediction[0] == 1 else "Not Placed"
placement_status

'Not Placed'

## Generate Placement Probability

The trained Gradient Boosting model is used to calculate the probability of the student's predicted placement status.

The probability is obtained using the model's `predict_proba()` method and converted into a percentage for display in the Streamlit application.

In [24]:
placement_probability = best_model.predict_proba(student_encoded)[0][1] * 100
placement_probability

np.float64(24.942759739099277)

In [25]:
round(placement_probability, 2)

np.float64(24.94)

## Career Recommendation

The career recommendation module evaluates the student's skills and experience against predefined career-specific requirements.

The system calculates a suitability score for each available career and identifies the most suitable career along with an alternative career.

The recommendation system considers coding skills, DSA score, machine learning knowledge, projects, internships, aptitude score, and communication skills.

In [26]:
career_weights = {
    'Data Scientist': {
        'coding_skills': 0.2,
        'dsa_score': 0.1,
        'ml_knowledge': 0.25,
        'projects_count': 0.2,
        'internships': 0.1,
        'aptitude_score': 0.1,
        'communication_skills': 0.05
    },

    'Data Analyst': {
        'coding_skills': 0.1,
        'dsa_score': 0.05,
        'ml_knowledge': 0.1,
        'projects_count': 0.15,
        'internships': 0.1,
        'aptitude_score': 0.25,
        'communication_skills': 0.25
    },

    'Machine Learning Engineer': {
        'coding_skills': 0.25,
        'dsa_score': 0.15,
        'ml_knowledge': 0.3,
        'projects_count': 0.15,
        'internships': 0.05,
        'aptitude_score': 0.05,
        'communication_skills': 0.05
    },

    'Software Developer': {
        'coding_skills': 0.3,
        'dsa_score': 0.25,
        'ml_knowledge': 0.05,
        'projects_count': 0.2,
        'internships': 0.05,
        'aptitude_score': 0.1,
        'communication_skills': 0.05
    },

    'Business Analyst': {
        'coding_skills': 0.05,
        'dsa_score': 0.05,
        'ml_knowledge': 0.05,
        'projects_count': 0.15,
        'internships': 0.1,
        'aptitude_score': 0.3,
        'communication_skills': 0.3
    }
}

In [27]:
career_scores = {}

for career, weights in career_weights.items():
    score = 0

    for feature, weight in weights.items():
        score += student_data[feature] * weight

    career_scores[career] = score

ranked_careers = sorted(
    career_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

ranked_careers

[('Machine Learning Engineer', 4.0),
 ('Data Analyst', 3.75),
 ('Software Developer', 3.75),
 ('Business Analyst', 3.75),
 ('Data Scientist', 3.5)]

In [28]:
recommended_career = ranked_careers[0][0]
career_suitability_score = ranked_careers[0][1]

alternative_career = ranked_careers[1][0]

{
    "recommended_career": recommended_career,
    "career_suitability_score": career_suitability_score,
    "alternative_career": alternative_career
}

{'recommended_career': 'Machine Learning Engineer',
 'career_suitability_score': 4.0,
 'alternative_career': 'Data Analyst'}